# 3 · Add models & compare — Kaggle

Incremental runner. Use this when some models were already scored (e.g. by
`2_run_all.ipynb`) and you only want to run the **new** ones, then merge and
re-do the full analysis.

Flow: run `config.PEERS` (Phi-3.5-mini, Qwen2.5-3B) → merge with a previously
saved `per_item.csv` → `analysis.analyze()` over all models (so the 5-model
ranking / agreement views become meaningful).

### Setup
1. **Accelerator → `GPU T4`**, **Internet → On**, add the **`HF_TOKEN`** secret.
2. Attach your earlier results as a **Dataset**: download `results/per_item.csv`
   from the previous run's Output, create a Kaggle Dataset from it, and **Add
   data** here. It will appear under `/kaggle/input/<name>/`. (Skip if you just
   want to score the new models on their own.)
3. **Run all.**

In [ ]:
REPO_URL = "https://github.com/ThuongHong/science-of-benchmarking-demo.git"
import os, sys, subprocess, glob
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
WORK = f"/kaggle/working/{REPO_DIR}"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, "src")
!pip install -q -U transformers accelerate bitsandbytes datasets sentencepiece
print("cwd:", os.getcwd())

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
print("HF login OK")

## 1. Previous results (optional)
Auto-discovers any `per_item*.csv` attached under `/kaggle/input`. Add paths to
`PREV_PATHS` manually if needed.

In [ ]:
import pandas as pd
PREV_PATHS = sorted(glob.glob("/kaggle/input/**/per_item*.csv", recursive=True))
print("found previous results:", PREV_PATHS or "(none)")
prev = pd.concat([pd.read_csv(p) for p in PREV_PATHS], ignore_index=True) \
    if PREV_PATHS else pd.DataFrame()
print("previous systems:", sorted(prev["system"].unique()) if len(prev) else [])

## 2. Build subsets + run the NEW models
`config.PEERS` = the models not in the previous run. Switch to `config.SYSTEMS`
for a full from-scratch run.

In [ ]:
import config, evaluate
RUN = config.PEERS          # new models only; use config.SYSTEMS for everything
print("running:", [s["name"] for s in RUN])
new = evaluate.run(RUN)
print(len(new), "new rows")

## 3. Merge + full analysis

In [ ]:
import analysis
merged = pd.concat([prev, new], ignore_index=True)
# if a (system, item) appears twice, keep the most recent run
merged = merged.drop_duplicates(subset=["system", "id"], keep="last")
merged.to_csv("results/per_item.csv", index=False)
print("merged systems:", sorted(merged["system"].unique()), "|", len(merged), "rows")
analysis.analyze(merged)

In [ ]:
from IPython.display import Image, display
print("Ranking on MMLU vs GSM8K (rank_shift>0 ⇒ benchmark changes the winner):")
display(pd.read_csv("results/ranking.csv"))
for fig in ["saturation", "model_agreement", "metric_gap", "robustness", "mmlu_by_subject"]:
    display(Image(f"results/figures/{fig}.png"))

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "results")
print("zipped -> /kaggle/working/results.zip  (download from the Output tab)")